# Reading sampled cells and downloading the related H5AD data from Census database 

In [1]:
import os
import shutil

import anndata as ad
import cellxgene_census
import numpy as np
import pandas as pd

import warnings
from anndata import ImplicitModificationWarning

census_version = '2025-11-08'


In [2]:
path = '/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE'

sample_id_path = os.path.join(path, "data", "iha_sample_ids.csv")
base_dir = os.path.join(path, "iha_celltypist")
os.makedirs(base_dir, exist_ok = True)

expression_path = os.path.join(base_dir, "iha_expression_raw.h5ad")
parts_dir = os.path.join(base_dir, "_iha_extract_parts")

chunk_size = 2000


In [3]:
# Defining paths, creating directories

sample_ids = pd.read_csv(sample_id_path)
expected_ids = np.sort(sample_ids["soma_joinid"].to_numpy(dtype = np.int64))


sample_meta = sample_ids.copy()
sample_meta["soma_joinid"] = sample_meta["soma_joinid"].astype("int64")
sample_meta = sample_meta.set_index("soma_joinid")

RESET_PARTS = False  # set True only if needed to download the data again

if RESET_PARTS and os.path.exists(parts_dir):
    shutil.rmtree(parts_dir)

os.makedirs(parts_dir, exist_ok = True)

part_paths = []

In [4]:
with cellxgene_census.open_soma(census_version = census_version) as census:
    for start in range(0, len(expected_ids), chunk_size):

        part_path = os.path.join(parts_dir, f"part_{start:05d}.h5ad")

        if os.path.exists(part_path):
            part_paths.append(part_path)
            print(f"Skipping downloaded part: {os.path.basename(part_path)}")
            continue

        coords = expected_ids[start:start + chunk_size]

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message = "Transforming to str index", category = ImplicitModificationWarning)
            adata_part = cellxgene_census.get_anndata(census, organism = "Homo sapiens", obs_coords = coords)
        
        # adata_part = cellxgene_census.get_anndata(census, organism = "Homo sapiens", obs_coords = coords)

        if "soma_joinid" not in adata_part.obs.columns:
            raise KeyError("Census does not contain obs['soma_joinid']")
            
        adata_part.obs_names = adata_part.obs["soma_joinid"].astype(str).to_numpy()

        got = pd.to_numeric(adata_part.obs["soma_joinid"], errors = "raise").to_numpy(dtype = np.int64)

        # Check every fragment before the append
        if len(got) != len(coords) or not np.array_equal(np.sort(got), coords):
            missing = np.setdiff1d(coords, got)
            extra = np.setdiff1d(got, coords)
            raise RuntimeError(f"ID mismatch in the chunk No. {start // chunk_size}: \n\
                Missing {missing[:10].tolist()}, \n\
                Extra {extra[:10].tolist()}")

        meta = sample_meta.reindex(pd.Index(got, name = "soma_joinid"))
        meta = meta.reset_index(drop = True)

        for col in meta.columns:
            if col in adata_part.obs.columns:
                census_values = adata_part.obs[col].reset_index(drop = True)
                if not census_values.astype("string").equals(meta[col].astype("string")):
                    raise ValueError(f"{col!r} in CSV mismatches with Census")
            else:
                adata_part.obs[col] = meta[col].to_numpy()

        part_path = os.path.join(parts_dir, f"part_{start:05d}.h5ad")
        adata_part.write_h5ad(part_path, compression = "gzip")
        part_paths.append(part_path)
        

        print(f"Received {min(start + len(coords), len(expected_ids)):,} out of {len(expected_ids):,} cells", flush = True)

if not part_paths:
    raise RuntimeError("Did not manage to get any part from Census")

Skipping downloaded part: part_00000.h5ad
Skipping downloaded part: part_02000.h5ad
Skipping downloaded part: part_04000.h5ad
Skipping downloaded part: part_06000.h5ad
Skipping downloaded part: part_08000.h5ad
Skipping downloaded part: part_10000.h5ad
Skipping downloaded part: part_12000.h5ad
Skipping downloaded part: part_14000.h5ad
Skipping downloaded part: part_16000.h5ad
Skipping downloaded part: part_18000.h5ad
Skipping downloaded part: part_20000.h5ad
Skipping downloaded part: part_22000.h5ad
Skipping downloaded part: part_24000.h5ad
Skipping downloaded part: part_26000.h5ad
Skipping downloaded part: part_28000.h5ad
Skipping downloaded part: part_30000.h5ad
Skipping downloaded part: part_32000.h5ad
Skipping downloaded part: part_34000.h5ad
Skipping downloaded part: part_36000.h5ad
Skipping downloaded part: part_38000.h5ad
Skipping downloaded part: part_40000.h5ad
Skipping downloaded part: part_42000.h5ad
Skipping downloaded part: part_44000.h5ad
Skipping downloaded part: part_460

In [5]:
part_paths = sorted(os.path.join(parts_dir, name) for name in os.listdir(parts_dir) if name.endswith(".h5ad"))

ad.experimental.concat_on_disk({f"part_{i:05d}": file for i, file in enumerate(part_paths)}, expression_path, 
                               axis = 0, join = "inner", merge = None)

In [6]:
check = ad.read_h5ad(expression_path, backed = "r")
try:
    extracted_ids = pd.to_numeric(check.obs["soma_joinid"], errors = "raise").to_numpy(dtype = np.int64)

    if len(extracted_ids) != len(expected_ids):
        raise AssertionError(f"The file contains {len(extracted_ids)} cells, {len(expected_ids)} expected")
    if len(np.unique(extracted_ids)) != len(extracted_ids):
        raise AssertionError("The final file contains repeated soma_joinid")
    if not np.array_equal(np.sort(extracted_ids), expected_ids):
        missing = np.setdiff1d(expected_ids, extracted_ids)
        extra = np.setdiff1d(extracted_ids, expected_ids)
        raise AssertionError(f"Final IDs do not match: \n\
            Missing {missing[:10].tolist()}, \n\
            Extra {extra[:10].tolist()}"
        )

    print(check)
    print(f"All {len(extracted_ids):,} chosen cells have matched")
finally:
    check.file.close()

print("Saved: ", expression_path)

AnnData object with n_obs × n_vars = 54225 × 61497 backed at '/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_raw.h5ad'
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'age', 'age_group'
All 54,225 chosen cells have matched
Saved:  /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_raw.h5ad
